# Caso 2: INTERMEDIO - Calidad de Datos de Ingresos Municipalidades (SIAF)

---

## Contexto del Proyecto

### Descripción del Problema
El **Ministerio de Economía y Finanzas (MEF)**, a través del Sistema Integrado de
Administración Financiera (SIAF), registra el presupuesto y ejecución de ingresos
de todos los niveles de gobierno. Al consolidar los datos históricos 2012-2026,
se detectaron inconsistencias en montos, registros de niveles de gobierno no
municipales mezclados, y valores faltantes en columnas clave que impiden
generar reportes presupuestales confiables.

### Objetivo Analítico
Limpiar y mejorar la calidad de los datos de ingresos históricos del SIAF para:
- Filtrar correctamente solo Gobiernos Locales (NIVEL_GOBIERNO = 'M')
- Imputar valores faltantes en montos usando la mediana por ejecutora y rubro
- Corregir tipos de dato y estandarizar códigos geográficos
- Construir el UBIGEO necesario para el JOIN con SISMEPRE y RENAMU
- Calcular KPIs básicos: TASA_EJECUCION y brecha PIA vs PIM
- Preparar el archivo Silver listo para el modelo estrella en Gold

### Impacto de la Mala Calidad de Datos
- **Financiero**: Montos de PIA, PIM o MONTO_RECAUDADO incorrectos llevan
a calcular mal el porcentaje de ejecución presupuestal por municipalidad
- **Operativo**: Registros de gobiernos nacionales o regionales mezclados
distorsionan los rankings y comparativos entre municipalidades
- **Estratégico**: Decisiones de política pública basadas en datos incompletos
pueden derivar en asignaciones presupuestales inequitativas entre municipios

---

## Dimensiones de Calidad a Corregir

En este caso aplicaremos correcciones sobre:

1. **Completitud**: Imputar nulos en montos con mediana por ejecutora y rubro
2. **Exactitud**: Corregir tipos de dato y montos negativos
3. **Consistencia**: Verificar coherencia entre NIVEL_GOBIERNO y su nombre
4. **Integridad**: Construir UBIGEO de 6 dígitos estandarizado
5. **Razonabilidad**: Tratar outliers usando IQR por rubro
6. **Oportunidad**: Verificar y documentar cobertura 2012-2026
7. **Unicidad**: Eliminar duplicados exactos y parciales por clave de negocio
8. **Validez**: Estandarizar códigos de RUBRO y FUENTE_FINANCIAMIENTO

---

---

# SOLUCIÓN NIVEL INTERMEDIO

## Objetivo
Recuperar datos cuando sea posible usando:
- Imputación inteligente de valores faltantes en montos presupuestales
- Corrección de montos negativos y tipos de dato incorrectos
- Detección de outliers en MONTO_PIM y MONTO_RECAUDADO con contexto de rubro
- Validaciones basadas en reglas del catálogo presupuestal del MEF
- Cálculo de KPIs básicos: TASA_EJECUCION y brecha presupuestal

### Filosofía:
"No eliminar sin antes intentar recuperar información valiosa —
un registro de municipalidad con nulo en MONTO_PIA puede imputarse
con la mediana de su mismo rubro y ejecutora en ese año"

In [6]:
# =========================================================
# LIBRERÍAS
# =========================================================

import pandas as pd
import numpy as np
import warnings
import gc

warnings.filterwarnings('ignore')

print("✅ Librerías cargadas correctamente")

✅ Librerías cargadas correctamente


In [7]:
# Cargar solo las columnas necesarias 
COLUMNAS_NECESARIAS = [
    'ANO_DOC',
    'MES_DOC',
    'NIVEL_GOBIERNO',
    'NIVEL_GOBIERNO_NOMBRE',
    'SEC_EJEC',
    'EJECUTORA',
    'EJECUTORA_NOMBRE',
    'DEPARTAMENTO_EJECUTORA',
    'DEPARTAMENTO_EJECUTORA_NOMBRE',
    'PROVINCIA_EJECUTORA',
    'PROVINCIA_EJECUTORA_NOMBRE',
    'DISTRITO_EJECUTORA',
    'DISTRITO_EJECUTORA_NOMBRE',
    'FUENTE_FINANCIAMIENTO',
    'FUENTE_FINANCIAMIENTO_NOMBRE',
    'RUBRO',
    'RUBRO_NOMBRE',
    'GENERICA',
    'GENERICA_NOMBRE',
    'SUBGENERICA',
    'SUBGENERICA_NOMBRE',
    'MONTO_PIA',
    'MONTO_PIM',
    'MONTO_RECAUDADO',
    'ARCHIVO_ORIGEN'
]

# Tipos optimizados para reducir RAM
TIPOS_DATOS = {
    'ANO_DOC': 'int16',
    'MES_DOC': 'int8',
    'NIVEL_GOBIERNO': 'category',
    'SEC_EJEC': 'string',
    'RUBRO': 'string',
    'FUENTE_FINANCIAMIENTO': 'string',
    'GENERICA': 'string',
    'SUBGENERICA': 'string',
    'MONTO_PIA': 'float32',
    'MONTO_PIM': 'float32',
    'MONTO_RECAUDADO': 'float32'
}

# Cargar dataset
df = pd.read_csv(
    'INGRESOS_HISTORICO_TOTAL_2012_2026.csv',
    encoding='latin-1',
    usecols=COLUMNAS_NECESARIAS,
    dtype=TIPOS_DATOS,
    low_memory=False
)

# Filtrar municipalidades inmediatamente
df = df[df['NIVEL_GOBIERNO'] == 'M'].copy()

print("✅ Dataset cargado correctamente")
print(f"📊 Registros municipalidades : {len(df):,}")
print(f"📅 Rango de años             : {df['ANO_DOC'].min()} - {df['ANO_DOC'].max()}")

# Consumo de memoria
memoria_mb = df.memory_usage(deep=True).sum() / 1024**2

print(f"📦 Memoria utilizada         : {memoria_mb:.2f} MB")

✅ Dataset cargado correctamente
📊 Registros municipalidades : 8,880,692
📅 Rango de años             : 2012 - 2026
📦 Memoria utilizada         : 3525.44 MB


In [26]:
# =========================================================
# SOLUCIÓN INTERMEDIA - CALIDAD DE DATOS
# =========================================================

import gc
import numpy as np
import pandas as pd

# =========================================================
# CONFIGURACIÓN GENERAL
# =========================================================

RANGO_ANIOS = (2012, 2026)

COLUMNAS_DUPLICADOS = [
    'ANO_DOC',
    'MES_DOC',
    'SEC_EJEC',
    'RUBRO',
    'FUENTE_FINANCIAMIENTO',
    'GENERICA',
    'SUBGENERICA'
]

COLUMNAS_MONTO = [
    'MONTO_PIA',
    'MONTO_PIM',
    'MONTO_RECAUDADO'
]

COLUMNAS_GEO = [
    'DEPARTAMENTO_EJECUTORA_NOMBRE',
    'PROVINCIA_EJECUTORA_NOMBRE',
    'DISTRITO_EJECUTORA_NOMBRE'
]

REGLAS_NIVEL = {
    'E': 'GOBIERNO NACIONAL',
    'R': 'GOBIERNOS REGIONALES',
    'M': 'GOBIERNOS LOCALES'
}

DEPARTAMENTOS_VALIDOS = [
    '01','02','03','04','05',
    '06','07','08','09','10',
    '11','12','13','14','15',
    '16','17','18','19','20',
    '21','22','23','24','25'
]

# =========================================================
# CREAR DATAFRAME DE TRABAJO
# =========================================================

df_intermedio = df.copy()

print("=" * 60)
print("      PIPELINE SILVER INTERMEDIO")
print("=" * 60)

print(f"\n📊 Registros iniciales: {len(df_intermedio):,}")

# =========================================================
# 1. CONVERSIÓN DE VARIABLES MONETARIAS
# =========================================================

print("\n🔹 1. Conversión de variables monetarias")

for col in COLUMNAS_MONTO:

    df_intermedio[col] = pd.to_numeric(
        df_intermedio[col],
        errors='coerce'
    )

print("✅ Variables monetarias convertidas")

# =========================================================
# 2. IMPUTACIÓN DE VALORES FALTANTES
# =========================================================

print("\n" + "=" * 60)
print("2. IMPUTACIÓN DE VALORES FALTANTES")
print("=" * 60)

# -----------------------------
# MONTO_PIA y MONTO_PIM
# -----------------------------

for col in ['MONTO_PIA', 'MONTO_PIM']:

    nulos = df_intermedio[col].isnull().sum()

    df_intermedio[col] = (
        df_intermedio
        .groupby(['RUBRO', 'ANO_DOC'])[col]
        .transform(
            lambda x: x.fillna(x.median())
        )
        .fillna(0)
    )

    print(f"✅ {col}: {nulos:,} valores imputados")

# -----------------------------
# MONTO_RECAUDADO
# -----------------------------

nulos_rec = (
    df_intermedio['MONTO_RECAUDADO']
    .isnull()
    .sum()
)

df_intermedio['MONTO_RECAUDADO'] = (
    df_intermedio['MONTO_RECAUDADO']
    .fillna(0)
)

print(f"✅ MONTO_RECAUDADO: {nulos_rec:,} valores imputados")

# -----------------------------
# EJECUTORA_NOMBRE
# -----------------------------

mask_ejec = (
    df_intermedio['EJECUTORA_NOMBRE']
    .isnull()
)

nulos_ejec = mask_ejec.sum()

df_intermedio.loc[
    mask_ejec,
    'EJECUTORA_NOMBRE'
] = (
    'EJECUTORA_' +
    df_intermedio.loc[
        mask_ejec,
        'SEC_EJEC'
    ].astype(str)
)

print(f"✅ EJECUTORA_NOMBRE: {nulos_ejec:,} valores imputados")

# =========================================================
# 3. CORRECCIÓN DE VALORES NEGATIVOS
# =========================================================

print("\n" + "=" * 60)
print("3. CORRECCIÓN DE VALORES NEGATIVOS")
print("=" * 60)

for col in COLUMNAS_MONTO:

    mask_negativos = (
        df_intermedio[col] < 0
    )

    cantidad_negativos = (
        mask_negativos.sum()
    )

    if cantidad_negativos > 0:

        medianas = (
            df_intermedio.loc[
                df_intermedio[col] >= 0
            ]
            .groupby(
                ['RUBRO', 'SEC_EJEC']
            )[col]
            .transform('median')
        )

        df_intermedio.loc[
            mask_negativos,
            col
        ] = (
            medianas[mask_negativos]
            .fillna(0)
        )

    print(
        f"✅ {col}: "
        f"{cantidad_negativos:,} negativos corregidos"
    )

# =========================================================
# 4. ESTANDARIZACIÓN DE FORMATOS
# =========================================================

print("\n" + "=" * 60)
print("4. ESTANDARIZACIÓN DE FORMATOS")
print("=" * 60)

# -----------------------------
# Variables geográficas
# -----------------------------

for col in COLUMNAS_GEO:

    df_intermedio[col] = (
        df_intermedio[col]
        .astype(str)
        .str.upper()
        .str.strip()
    )

# -----------------------------
# RUBRO_NOMBRE
# -----------------------------

df_intermedio['RUBRO_NOMBRE'] = (
    df_intermedio['RUBRO_NOMBRE']
    .astype(str)
    .str.upper()
    .str.strip()
)

print("✅ Variables estandarizadas")

# =========================================================
# 5. CORRECCIÓN DE INCONSISTENCIAS
# =========================================================

print("\n" + "=" * 60)
print("5. CORRECCIÓN DE INCONSISTENCIAS")
print("=" * 60)

df_intermedio['NIVEL_GOBIERNO_NOMBRE'] = (
    df_intermedio['NIVEL_GOBIERNO']
    .map(REGLAS_NIVEL)
)

print("✅ NIVEL_GOBIERNO_NOMBRE corregido")

# =========================================================
# 6. CREACIÓN DE UBIGEO
# =========================================================

print("\n🔹 6. Creación de UBIGEO")

# ---------------------------------------------------------
# Limpiar códigos geográficos
# ---------------------------------------------------------

COLUMNAS_CODIGO_GEO = [
    'DEPARTAMENTO_EJECUTORA',
    'PROVINCIA_EJECUTORA',
    'DISTRITO_EJECUTORA'
]

for col in COLUMNAS_CODIGO_GEO:

    df_intermedio[col] = (
        df_intermedio[col]
        .astype(str)
        .str.strip()
        .replace(['', 'nan', 'None'], np.nan)
    )

# ---------------------------------------------------------
# Completar nulos geográficos
# ---------------------------------------------------------

for col in COLUMNAS_CODIGO_GEO:

    df_intermedio[col] = (
        df_intermedio[col]
        .fillna('00')
        .str.zfill(2)
    )

# ---------------------------------------------------------
# Crear UBIGEO
# ---------------------------------------------------------

df_intermedio['UBIGEO'] = (
    df_intermedio['DEPARTAMENTO_EJECUTORA']
    +
    df_intermedio['PROVINCIA_EJECUTORA']
    +
    df_intermedio['DISTRITO_EJECUTORA']
)

# ---------------------------------------------------------
# Validar departamentos correctos
# ---------------------------------------------------------

departamentos_validos = [
    '01','02','03','04','05',
    '06','07','08','09','10',
    '11','12','13','14','15',
    '16','17','18','19','20',
    '21','22','23','24','25'
]

mask_dep = (
    df_intermedio['UBIGEO']
    .str[:2]
    .isin(departamentos_validos)
)

print(f"✅ UBIGEO con departamento válido: "
      f"{mask_dep.sum():,}")

# =========================================================
# 7. KPI — TASA DE EJECUCIÓN
# =========================================================

print("\n🔹 7. Cálculo de KPI")

df_intermedio['TASA_EJECUCION'] = np.where(
    df_intermedio['MONTO_PIM'] > 0,
    (
        df_intermedio['MONTO_RECAUDADO']
        / df_intermedio['MONTO_PIM']
        * 100
    ).round(2),
    0
)

# Eliminar posibles NaN restantes
df_intermedio['TASA_EJECUCION'] = (
    df_intermedio['TASA_EJECUCION']
    .fillna(0)
)

print(
    f"✅ Tasa promedio: "
    f"{df_intermedio['TASA_EJECUCION'].mean():.2f}%"
)

# =========================================================
# 8. MANEJO DE DUPLICADOS
# =========================================================

print("\n" + "=" * 60)
print("8. MANEJO DE DUPLICADOS")
print("=" * 60)

# -----------------------------
# Duplicados exactos
# -----------------------------

columnas_exactas = [
    col for col in df_intermedio.columns
    if col != 'ARCHIVO_ORIGEN'
]

antes_exactos = len(df_intermedio)

df_intermedio = df_intermedio.drop_duplicates(
    subset=columnas_exactas,
    keep='first'
)

eliminados_exactos = (
    antes_exactos - len(df_intermedio)
)

print(
    f"✅ Duplicados exactos eliminados: "
    f"{eliminados_exactos:,}"
)

# -----------------------------
# Duplicados de negocio
# -----------------------------

antes_negocio = len(df_intermedio)

df_intermedio = df_intermedio.drop_duplicates(
    subset=COLUMNAS_DUPLICADOS,
    keep='first'
)

eliminados_negocio = (
    antes_negocio - len(df_intermedio)
)

print(
    f"✅ Duplicados negocio eliminados: "
    f"{eliminados_negocio:,}"
)

# =========================================================
# 9. DETECCIÓN DE OUTLIERS
# =========================================================

print("\n" + "=" * 60)
print("9. DETECCIÓN DE OUTLIERS")
print("=" * 60)

for col in ['MONTO_PIM', 'MONTO_RECAUDADO']:

    Q1 = (
        df_intermedio[col]
        .quantile(0.25)
    )

    Q3 = (
        df_intermedio[col]
        .quantile(0.75)
    )

    IQR = Q3 - Q1

    limite_superior = (
        Q3 + (6 * IQR)
    )

    outliers = (
        df_intermedio[col]
        > limite_superior
    )

    cantidad_outliers = (
        outliers.sum()
    )

    # Crear bandera
    df_intermedio[
        f'OUTLIER_{col}'
    ] = np.where(
        outliers,
        1,
        0
    )

    print(
        f"✅ {col}: "
        f"{cantidad_outliers:,} outliers detectados"
    )

# =========================================================
# 10. VALIDACIÓN TEMPORAL
# =========================================================

print("\n🔹 10. Validación temporal")

antes_anios = len(df_intermedio)

df_intermedio = df_intermedio[
    (
        df_intermedio['ANO_DOC']
        >= RANGO_ANIOS[0]
    )
    &
    (
        df_intermedio['ANO_DOC']
        <= RANGO_ANIOS[1]
    )
]

eliminados_anio = (
    antes_anios - len(df_intermedio)
)

print(
    f"✅ Registros fuera de rango: "
    f"{eliminados_anio:,}"
)

# =========================================================
# LIMPIEZA FINAL DE NULOS
# =========================================================

print("\n🔹 11. Limpieza final de nulos")

nulos_antes = (
    df_intermedio
    .isnull()
    .sum()
    .sum()
)

df_intermedio[COLUMNAS_MONTO] = (
    df_intermedio[COLUMNAS_MONTO]
    .fillna(0)
)

df_intermedio['TASA_EJECUCION'] = (
    df_intermedio['TASA_EJECUCION']
    .fillna(0)
)

nulos_despues = (
    df_intermedio
    .isnull()
    .sum()
    .sum()
)

print(
    f"✅ Nulos antes : {nulos_antes:,}"
)

print(
    f"✅ Nulos después : {nulos_despues:,}"
)

# =========================================================
# RESUMEN FINAL
# =========================================================

perdida_total = (
    len(df) - len(df_intermedio)
)

porcentaje_final = (
    len(df_intermedio)
    / len(df)
    * 100
)

print("\n" + "=" * 60)
print("               RESUMEN FINAL")
print("=" * 60)

print(f"\n📊 Registros iniciales : {len(df):,}")

print(f"📊 Registros finales   : {len(df_intermedio):,}")

print(
    f"📉 Pérdida total       : "
    f"{perdida_total:,} registros "
    f"({perdida_total / len(df) * 100:.2f}%)"
)

print(
    f"📈 Registros conservados : "
    f"{porcentaje_final:.2f}%"
)

# =========================================================
# LIBERAR MEMORIA
# =========================================================

gc.collect()

print("\n✅ Pipeline Silver intermedio completado")

      PIPELINE SILVER INTERMEDIO

📊 Registros iniciales: 8,880,692

🔹 1. Conversión de variables monetarias
✅ Variables monetarias convertidas

2. IMPUTACIÓN DE VALORES FALTANTES
✅ MONTO_PIA: 0 valores imputados
✅ MONTO_PIM: 0 valores imputados
✅ MONTO_RECAUDADO: 0 valores imputados
✅ EJECUTORA_NOMBRE: 0 valores imputados

3. CORRECCIÓN DE VALORES NEGATIVOS
✅ MONTO_PIA: 0 negativos corregidos
✅ MONTO_PIM: 13,503 negativos corregidos
✅ MONTO_RECAUDADO: 62,726 negativos corregidos

4. ESTANDARIZACIÓN DE FORMATOS
✅ Variables estandarizadas

5. CORRECCIÓN DE INCONSISTENCIAS
✅ NIVEL_GOBIERNO_NOMBRE corregido

🔹 6. Creación de UBIGEO
✅ UBIGEO con departamento válido: 8,880,656

🔹 7. Cálculo de KPI
✅ Tasa promedio: 35.50%

8. MANEJO DE DUPLICADOS
✅ Duplicados exactos eliminados: 474,986
✅ Duplicados negocio eliminados: 5,553,025

9. DETECCIÓN DE OUTLIERS
✅ MONTO_PIM: 544,368 outliers detectados
✅ MONTO_RECAUDADO: 448,094 outliers detectados

🔹 10. Validación temporal
✅ Registros fuera de rang

In [30]:
# =========================================================
# MÓDULO 4 — VERIFICACIÓN POST-LIMPIEZA
# =========================================================

print("\n" + "=" * 60)
print("        VERIFICACIÓN POST-LIMPIEZA")
print("   Ingresos Municipalidades SIAF 2012-2026")
print("=" * 60)

# =========================================================
# MÉTRICAS GENERALES
# =========================================================

total_registros = len(df_intermedio)

total_nulos = (
    df_intermedio
    .isnull()
    .sum()
    .sum()
)

duplicados_negocio = (
    df_intermedio
    .duplicated(subset=COLUMNAS_DUPLICADOS)
    .sum()
)

# =========================================================
# VALIDACIÓN DE MONTOS NEGATIVOS
# =========================================================

pia_negativos = (
    (df_intermedio['MONTO_PIA'] < 0)
    .sum()
)

pim_negativos = (
    (df_intermedio['MONTO_PIM'] < 0)
    .sum()
)

recaudado_negativos = (
    (df_intermedio['MONTO_RECAUDADO'] < 0)
    .sum()
)

# =========================================================
# VALIDACIÓN DE UBIGEO
# =========================================================

ubigeos_validos = (
    df_intermedio['UBIGEO']
    .str.len()
    .eq(6)
    .sum()
)

# =========================================================
# VALIDACIÓN KPI
# =========================================================

tasa_ejecucion_validos = (
    df_intermedio['TASA_EJECUCION']
    .notna()
    .sum()
)

# =========================================================
# COBERTURA TEMPORAL
# =========================================================

anio_minimo = (
    df_intermedio['ANO_DOC']
    .min()
)

anio_maximo = (
    df_intermedio['ANO_DOC']
    .max()
)

# =========================================================
# ENTIDADES ÚNICAS
# =========================================================

total_municipalidades = (
    df_intermedio['SEC_EJEC']
    .nunique()
)

total_departamentos = (
    df_intermedio['DEPARTAMENTO_EJECUTORA']
    .nunique()
)

# =========================================================
# VALIDACIÓN DE CATEGORÍAS
# =========================================================

niveles_gobierno = list(
    df_intermedio['NIVEL_GOBIERNO']
    .unique()
)

fuentes_financiamiento = sorted(
    df_intermedio['FUENTE_FINANCIAMIENTO']
    .astype(str)
    .unique()
)

# =========================================================
# VALIDACIÓN DE NULOS POR COLUMNA
# =========================================================

nulos_por_columna = (
    df_intermedio
    .isnull()
    .sum()
)

nulos_por_columna = (
    nulos_por_columna[nulos_por_columna > 0]
    .sort_values(ascending=False)
)

# =========================================================
# VALIDACIÓN DE UBIGEO POR DEPARTAMENTO
# =========================================================

departamentos_validos = [
    '01','02','03','04','05',
    '06','07','08','09','10',
    '11','12','13','14','15',
    '16','17','18','19','20',
    '21','22','23','24','25'
]

departamentos_detectados = sorted(
    df_intermedio['UBIGEO']
    .str[:2]
    .unique()
)

departamentos_invalidos = sorted(
    list(
        set(departamentos_detectados)
        - set(departamentos_validos)
    )
)

# =========================================================
# REPORTE FINAL
# =========================================================

print(f"\n📊 Total registros                 : {total_registros:,}")

print(f"✅ Valores nulos totales           : {total_nulos:,}")

# ---------------------------------------------------------
# Mostrar columnas con nulos
# ---------------------------------------------------------

if len(nulos_por_columna) > 0:

    print("\n📌 Columnas con valores nulos:")
    print(nulos_por_columna)

else:

    print("\n✅ No existen valores nulos")

# ---------------------------------------------------------

print(f"\n✅ Duplicados clave negocio        : "
      f"{duplicados_negocio:,}")

print(f"✅ MONTO_PIA negativos             : "
      f"{pia_negativos:,}")

print(f"✅ MONTO_PIM negativos             : "
      f"{pim_negativos:,}")

print(f"✅ MONTO_RECAUDADO negativos       : "
      f"{recaudado_negativos:,}")

print(f"✅ UBIGEO válidos                  : "
      f"{ubigeos_validos:,}")

print(f"✅ TASA_EJECUCION calculada        : "
      f"{tasa_ejecucion_validos:,}")

print("\n" + "-" * 60)

print(f"📅 Cobertura temporal              : "
      f"{anio_minimo} - {anio_maximo}")

print(f"🏘️ Municipalidades distintas       : "
      f"{total_municipalidades:,}")

departamentos_validos_count = (
    df_intermedio[
        df_intermedio['DEPARTAMENTO_EJECUTORA'] != '00'
    ]['DEPARTAMENTO_EJECUTORA']
    .nunique()
)

print(f"🗺️ Departamentos válidos           : "
      f"{departamentos_validos_count:,}")
print(departamentos_detectados)
if '00' in departamentos_detectados:

    registros_sin_departamento = (
        df_intermedio['DEPARTAMENTO_EJECUTORA']
        .eq('00')
        .sum()
    )

    print("\n⚠️ Registros con departamento desconocido:")
    print(f"   {registros_sin_departamento:,}")

# ---------------------------------------------------------
# Departamentos UBIGEO
# ---------------------------------------------------------

print("\n📌 Departamentos UBIGEO detectados:")
print(departamentos_detectados)

if '00' in departamentos_detectados:
    registros_sin_departamento = (
        df_intermedio['DEPARTAMENTO_EJECUTORA']
        .eq('00')
        .sum()
    )
    print("\n⚠️ Registros con departamento desconocido:")
    print(f"   {registros_sin_departamento:,}")

else:
    print("\n✅ Todos los departamentos son válidos")

print("\n" + "-" * 60)

print("📌 Valores únicos NIVEL_GOBIERNO:")
print(f"   {niveles_gobierno}")

print("\n📌 Fuentes de financiamiento:")
print(f"   {fuentes_financiamiento}")

print("\n✅ Verificación completada correctamente")


        VERIFICACIÓN POST-LIMPIEZA
   Ingresos Municipalidades SIAF 2012-2026

📊 Total registros                 : 2,852,681
✅ Valores nulos totales           : 0

✅ No existen valores nulos

✅ Duplicados clave negocio        : 0
✅ MONTO_PIA negativos             : 0
✅ MONTO_PIM negativos             : 0
✅ MONTO_RECAUDADO negativos       : 0
✅ UBIGEO válidos                  : 2,852,681
✅ TASA_EJECUCION calculada        : 2,852,681

------------------------------------------------------------
📅 Cobertura temporal              : 2012 - 2026
🏘️ Municipalidades distintas       : 1,964
🗺️ Departamentos válidos           : 25
['00', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25']

⚠️ Registros con departamento desconocido:
   25

📌 Departamentos UBIGEO detectados:
['00', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', 

In [31]:
# =========================================================
# MÓDULO 5 — EXPORTACIÓN SILVER INTERMEDIO
# =========================================================

print("\n" + "=" * 60)
print("          EXPORTACIÓN DATASET SILVER")
print("=" * 60)

# =========================================================
# COLUMNAS A CONVERTIR
# =========================================================

COLUMNAS_STRING = [
    'DEPARTAMENTO_EJECUTORA',
    'PROVINCIA_EJECUTORA',
    'DISTRITO_EJECUTORA',
    'RUBRO',
    'FUENTE_FINANCIAMIENTO',
    'TIPO_RECURSO',
    'GENERICA',
    'SUBGENERICA',
    'SEC_EJEC'
]

# =========================================================
# CONVERSIÓN DE TIPOS
# =========================================================

print("\n🔹 Convirtiendo columnas a string")

for col in COLUMNAS_STRING:

    if col in df_intermedio.columns:

        df_intermedio[col] = (
            df_intermedio[col]
            .astype(str)
        )

        print(f"✅ {col}")

# =========================================================
# EXPORTACIÓN PARQUET
# =========================================================

nombre_archivo = 'ingresos_silver_intermedio.parquet'

df_intermedio.to_parquet(
    nombre_archivo,
    index=False
)

# =========================================================
# RESUMEN FINAL
# =========================================================

print("\n" + "-" * 60)

print(f"✅ Archivo exportado correctamente")

print(f"\n📁 Nombre archivo : {nombre_archivo}")

print(f"📊 Total registros: {len(df_intermedio):,}")

print(f"📋 Total columnas : {len(df_intermedio.columns):,}")

print("\n✅ Proceso Silver intermedio finalizado")


          EXPORTACIÓN DATASET SILVER

🔹 Convirtiendo columnas a string
✅ DEPARTAMENTO_EJECUTORA
✅ PROVINCIA_EJECUTORA
✅ DISTRITO_EJECUTORA
✅ RUBRO
✅ FUENTE_FINANCIAMIENTO
✅ GENERICA
✅ SUBGENERICA
✅ SEC_EJEC

------------------------------------------------------------
✅ Archivo exportado correctamente

📁 Nombre archivo : ingresos_silver_intermedio.parquet
📊 Total registros: 2,852,681
📋 Total columnas : 29

✅ Proceso Silver intermedio finalizado


### Conclusiones de la Solución Intermedia

**Ventajas:**
- Recupera registros de municipalidades mediante imputación inteligente
  por RUBRO y ANO_DOC, preservando la distribución real de cada rubro
- Aplica reglas del catálogo presupuestal del MEF para corregir
  inconsistencias entre códigos y nombres
- Construye el UBIGEO estandarizado necesario para el JOIN con
  SISMEPRE y RENAMU en la integración Silver final
- Calcula TASA_EJECUCION como KPI base para los dashboards de Gold

**Desventajas:**
- La imputación por mediana de rubro puede no reflejar la realidad
  de municipalidades pequeñas con muy pocos registros en ese rubro
- No detecta errores de registro en el sistema SIAF (datos correctos
  formalmente pero incorrectos en la realidad)
- No automatiza validaciones contra el catálogo completo del MEF